https://www.kaggle.com/code/jaewook704/marvel-network

In [ ]:

import numpy as np
import pandas as pd
from itertools import combinations
from collections import Counter

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

from IPython.display import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from PIL import Image

import networkx as nx # sosyal ağ analizi için

from sklearn.preprocessing import MinMaxScaler
import random
import pickle

TEMPLATE = 'simple_white'

In [2]:
edge_df = pd.read_csv("edges.csv")
node_df = pd.read_csv("nodes.csv")
hero_net_df = pd.read_csv("hero-network.csv")

In [3]:
temp = pd.DataFrame({'edges.csv':sorted([h for h in edge_df['hero'].unique() if 'SPIDER' in h]),
                     'hero-network.csv':sorted([h for h in hero_net_df['hero1'].unique() if 'SPIDER' in h])})

display(temp)

for c in ['hero1', 'hero2']:
    print(f"{c} name max in hero-network.csv : {max(hero_net_df[c].apply(lambda x : len(x)))}")
    

# Name Preprocessing
## only use len 20 & only use left string bas on "/"
for c in ['hero1', 'hero2']:
    hero_net_df[c] = hero_net_df[c].apply(lambda x : x[:20].split("/")[0])
edge_df['hero'] = edge_df['hero'].apply(lambda x : x[:20].split("/")[0])

,edges.csv,hero-network.csv
0,"BEACH, SPIDER","BEACH, SPIDER"
1,BLOOD SPIDER/,BLOOD SPIDER/
2,MAN-SPIDER CLONE | M,MAN-SPIDER CLONE | M
3,MAN-SPIDER | MUTANT,MAN-SPIDER | MUTANT
4,SPIDER-MAN CLONE/BEN,SPIDER-MAN CLONE/BEN
5,SPIDER-MAN III/MARTH,SPIDER-MAN III/MARTH
6,SPIDER-MAN/PETER PARKER,SPIDER-MAN/PETER PAR
7,SPIDER-WOMAN DOPPELG,SPIDER-WOMAN DOPPELG
8,SPIDER-WOMAN II/JULI,SPIDER-WOMAN II/JULI
9,SPIDER-WOMAN IV/CHAR,SPIDER-WOMAN IV/CHAR


hero1 name max in hero-network.csv : 20
hero2 name max in hero-network.csv : 20


In [4]:
print("SPIDER & HULK in hero-network.csv")
print(f"hero1=SPIDER-MAN, hero2=HULK : {len(hero_net_df[(hero_net_df['hero1']=='SPIDER-MAN')&(hero_net_df['hero2']=='HULK')])}")
print(f"hero1=HULK, hero2=SPIDER-MAN : {len(hero_net_df[(hero_net_df['hero2']=='SPIDER-MAN')&(hero_net_df['hero1']=='HULK')])}")

temp1 = set(edge_df[edge_df['hero']=='SPIDER-MAN']['comic'])
temp2 = set(edge_df[edge_df['hero']=='HULK']['comic'])
print(f"Intersection in edges.csv : {len(temp1.intersection(temp2))}")

SPIDER & HULK in hero-network.csv
hero1=SPIDER-MAN, hero2=HULK : 43
hero1=HULK, hero2=SPIDER-MAN : 50
Intersection in edges.csv : 93


In [5]:
topn = 25
topn_hero = edge_df.groupby(['hero'])[['comic']].count().sort_values(by=['comic'], ascending=False).head(topn).index

h1_ = []; h2_ = []; cnt_ = [];
for comb in list(combinations(topn_hero, 2)):    
    temp1 = set(edge_df[edge_df['hero']==comb[0]]['comic'])
    temp2 = set(edge_df[edge_df['hero']==comb[1]]['comic'])
    cnt = len(temp1.intersection(temp2)) # Appear Together    
    h1_.append(comb[0]); h2_.append(comb[1]); cnt_.append(cnt);
appto_df = pd.DataFrame({'H1':h1_, 'H2':h2_, 'CNT':cnt_})

display(appto_df.head())

,H1,H2,CNT
0,SPIDER-MAN,CAPTAIN AMERICA,145
1,SPIDER-MAN,IRON MAN,95
2,SPIDER-MAN,THING,125
3,SPIDER-MAN,THOR,96
4,SPIDER-MAN,HUMAN TORCH,147


In [6]:
# 25 KAHRAMANIN AĞI
HERO_COLOR = {
    'CAPTAIN AMERICA':'darkblue',
    'IRON MAN':'gold',
    'SPIDER-MAN':'darkred',
    'HULK':'forestgreen',
    'THOR':'lightblue',
    'DR. STRANGE':'purple'
}

# Make network
## Initialize graph
## - https://towardsdatascience.com/tutorial-network-visualization-basics-with-networkx-and-plotly-and-a-little-nlp-57c9bbb55bb9
marvel_net = nx.Graph() 
for i, row in appto_df.iterrows():
    marvel_net.add_edge(row['H1'], row['H2'], weight=row['CNT'])  # specify edge data

## - Inference
# aspl = nx.average_shortest_path_length(marvel_net) # no weight
# adgr = sum(dict(marvel_net.degree()).values())/float(len(marvel_net)) # no weight
        
# Visualization
## Get positions for the nodes in network
# pos_ = nx.kamada_kawai_layout(marvel_net) # sample layout : spring_layout ...
pos_ = nx.spring_layout(marvel_net, seed=11)
cent_ = nx.pagerank(marvel_net, weight='weight') # page rank
cent_top = sorted(cent_.items(), key=lambda item: item[1], reverse=True)[:1] # page rank top 1

## Custom function to create an edge between node x and node y, with a given text and width
def make_edge(x, y, text, width):
    return  go.Scatter(x=x, y=y, line=dict(width=width, color='lightgray'), hoverinfo='text', text=([text]), mode='lines')

## For each edge, make an edge_trace, append to list
edge_trace = []
for edge in marvel_net.edges():    
    if marvel_net.edges()[edge]['weight'] > 0:
        char_1 = edge[0]
        char_2 = edge[1]
        x0, y0 = pos_[char_1]
        x1, y1 = pos_[char_2]
        trace  = make_edge([x0, x1, None], [y0, y1, None], None, width=5*(marvel_net.edges()[edge]['weight']/appto_df['CNT'].max()))
        edge_trace.append(trace)
                
## Make a node trace
node_trace = go.Scatter(x=[], y=[], text=[], textposition="top center", textfont_size=10, mode='markers+text', hoverinfo='none',
                        marker=dict(color=[], size=[], line_width=[], line_color=[]))

## For each node in network, get the position and size and add to the node_trace
for node in marvel_net.nodes():
    x, y = pos_[node]
    node_trace['x'] += tuple([x])
    node_trace['y'] += tuple([y])
    color = 'gray'
    line_width = 2
    line_color = 'darkgray'
    name_text = node
    
    if node in HERO_COLOR:
        color = HERO_COLOR[node]; line_color='black';
        
    if node in [v[0] for v in cent_top]:
        name_text = '<b>' + node + '</b>'
        
    node_trace['marker']['color'] += tuple([color])
    node_trace['marker']['size'] += tuple([int(400*cent_[node])]) # node size is proportional to page rank
    node_trace['marker']['line_width'] += tuple([line_width])
    node_trace['marker']['line_color'] += tuple([line_color])
    node_trace['text'] += tuple([name_text])
    
    
## Customize layout
layout = go.Layout(
    paper_bgcolor='rgba(0,0,0,0)', # transparent background
    plot_bgcolor='rgba(0,0,0,0)', # transparent 2nd background
    xaxis =  {'showgrid': False, 'zeroline': False}, # no gridlines
    yaxis = {'showgrid': False, 'zeroline': False}, # no gridlines
)

## Create figure
fig = go.Figure(layout = layout)
## Add all edge traces
for trace in edge_trace:
    fig.add_trace(trace)
fig.add_trace(node_trace)
fig.update_layout(showlegend = False)
fig.update_xaxes(showticklabels = False)
fig.update_yaxes(showticklabels = False)
fig.update_layout(title=f"<b>Top {topn} Heroes Network</b>")
fig.show()

Based on the centrality of the network, let's check who is close to the captain. I checked 5 centrality for 25 heroes. Since there is a weight on the edge, I reflected the weight in the centrality calculation.

PageRank : Pagerank computes a ranking of the nodes in the graph G based on the structure of the incoming links. It was originally designed as an algorithm to rank web pages. (networkx)

Eigenvector Centrality : Eigenvector centrality computes the centrality for a node based on the centrality of its neighbors. (networkx)

Degree Centrality : The degree centrality for a node v is the fraction of nodes it is connected to. (networkx)

Closeness Centrality : Closeness centrality of a node u is the reciprocal of the average shortest path distance to u over all n-1 reachable nodes. (networkx)

Betweenness Centrality : Betweenness centrality of a node v is the sum of the fraction of all-pairs shortest paths that pass through v. (networkx)

In [7]:
print(f"The number of hero pairs that never came out together : {len(appto_df[appto_df['CNT']==0])}")

cent_df = pd.DataFrame(index=list(marvel_net.nodes()))

# pagerank
cent_ = nx.pagerank(marvel_net, weight='weight')
cent_df['w_pagerank_cent'] = pd.Series(index=[k for k, v in cent_.items()], data=[float(v) for k, v in cent_.items()])

# eigenvalue centrality
cent_ = nx.eigenvector_centrality(marvel_net, weight='weight')
cent_df['w_eigenvector_cent'] = pd.Series(index=[k for k, v in cent_.items()], data=[float(v) for k, v in cent_.items()])

# degree centrality
cent_ = {h:0.0 for h in marvel_net.nodes()}
for u, v, d in marvel_net.edges(data=True):
    cent_[u]+=d['weight']; cent_[v]+=d['weight'];
cent_df['w_degree_cent'] = pd.Series(index=[k for k, v in cent_.items()], data=[float(v) for k, v in cent_.items()])

# closeness centrality
temp_net = marvel_net.copy()
for u,v,d in temp_net.edges(data=True):
    if 'distance' not in d:
        d['distance'] = 1.0/d['weight']
cent_ = nx.closeness_centrality(temp_net, distance='distance')
cent_df['w_closeness_cent'] = pd.Series(index=[k for k, v in cent_.items()], data=[float(v) for k, v in cent_.items()])

# betweenness centrality
cent_ = nx.betweenness_centrality(marvel_net, weight='weight')
cent_df['w_betweenness_cent'] = pd.Series(index=[k for k, v in cent_.items()], data=[float(v) for k, v in cent_.items()])

display(cent_df)
cent_df = cent_df.drop(columns=['w_betweenness_cent'])

The number of hero pairs that never came out together : 0


,w_pagerank_cent,w_eigenvector_cent,w_degree_cent,w_closeness_cent,w_betweenness_cent
SPIDER-MAN,0.050127,0.145800,2874.0,95.334005,0.000000
CAPTAIN AMERICA,0.066699,0.312066,4409.0,141.338880,0.000000
IRON MAN,0.054943,0.271015,3600.0,118.305766,0.000000
THING,0.057688,0.318561,3828.0,104.086414,0.000000
THOR,0.046146,0.231617,2969.0,108.986186,0.000000
HUMAN TORCH,0.056935,0.316594,3776.0,102.769509,0.000000
MR. FANTASTIC,0.055603,0.313944,3689.0,99.228022,0.000000
HULK,0.029614,0.112058,1674.0,79.893904,0.000000
WOLVERINE,0.033988,0.099205,1929.0,71.713747,0.000000
INVISIBLE WOMAN,0.052618,0.301620,3478.0,96.400542,0.000000


Based on the centrality results earlier, I think we should do the following.

Because all heroes are interconnected, Betweenness Centrality, which considers the shortest distance of all pairs, is not appropriate.
Because each centrality has a different scale, it is necessary to unify the range for comparison. I used MinMaxScaler.
After checking the mean centrality, Captain America was the captain!

Note: Here, only the frequent appearance of other heroes was considered, but other qualities as a captain were not considered.
Thor and Spider-Man have average centrality, Iron Man has high centrality, and Hulk and Dr. Strange have low centrality.
This mean of centrality seems to be similar to the character centrality I felt in the movie.

In [8]:
# Scaling
for c in cent_df.columns:
    s = MinMaxScaler()
    cent_df[[c]] = s.fit_transform(cent_df[[c]])  
cent_df['mean_cent'] = cent_df.mean(axis=1)
cent_df = cent_df.sort_values(by=['mean_cent'], ascending=False)

# Visualization
fig = go.Figure(data=[go.Bar(
    x=cent_df.index,
    y=cent_df['mean_cent'],
    marker_color=[HERO_COLOR['CAPTAIN AMERICA']]+['lightgray']*2+\
                 [HERO_COLOR['IRON MAN']]+['lightgray']*5+\
                 [HERO_COLOR['THOR']]+['lightgray']*2+\
                 [HERO_COLOR['SPIDER-MAN']]+['lightgray']*3+\
                 [HERO_COLOR['HULK']]+['lightgray']*5+\
                 [HERO_COLOR['DR. STRANGE']]+['lightgray']*2
)])
fig.update_layout(title_text='<b>Mean Centrality of Heros</b>', template=TEMPLATE)

In [9]:
def show_network(input_, title=""):
    input_net = input_.copy()
    
    HERO_COLOR = {
        'CAPTAIN AMERICA':'darkblue',
        'IRON MAN':'gold',
        'SPIDER-MAN':'darkred',
        'HULK':'forestgreen',
        'THOR':'lightblue',
        'DR. STRANGE':'purple'
    }

    # Visualization
    ## Get positions for the nodes in network
    pos_ = nx.spring_layout(input_net, seed=11)
    cent_ = nx.pagerank(input_net, weight='weight') # page rank
    cent_top = sorted(cent_.items(), key=lambda item: item[1], reverse=True)[:1] # page rank top 1

    ## Custom function to create an edge between node x and node y, with a given text and width
    def make_edge(x, y, text, width):
        return  go.Scatter(x=x, y=y, line=dict(width=width, color='lightgray'), hoverinfo='text', text=([text]), mode='lines')

    ## For each edge, make an edge_trace, append to list
    edge_trace = []
    for edge in input_net.edges():    
        if input_net.edges()[edge]['weight'] > 0:
            char_1 = edge[0]
            char_2 = edge[1]
            x0, y0 = pos_[char_1]
            x1, y1 = pos_[char_2]
            trace  = make_edge([x0, x1, None], [y0, y1, None], None, width=5*(input_net.edges()[edge]['weight']/appto_df['CNT'].max()))
            edge_trace.append(trace)

    ## Make a node trace
    node_trace = go.Scatter(x=[], y=[], text=[], textposition="top center", textfont_size=10, mode='markers+text', hoverinfo='none',
                            marker=dict(color=[], size=[], line_width=[], line_color=[]))

    ## For each node in network, get the position and size and add to the node_trace
    for node in input_net.nodes():
        x, y = pos_[node]
        node_trace['x'] += tuple([x])
        node_trace['y'] += tuple([y])
        color = 'gray'
        line_width = 2
        line_color = 'darkgray'
        name_text = ''

        if node in HERO_COLOR:
            color = HERO_COLOR[node]; line_color='black';
            name_text = node

        node_trace['marker']['color'] += tuple([color])
        node_trace['marker']['size'] += tuple([int(400*cent_[node])]) # node size is proportional to page rank
        node_trace['marker']['line_width'] += tuple([line_width])
        node_trace['marker']['line_color'] += tuple([line_color])
        node_trace['text'] += tuple([name_text])


    ## Customize layout
    layout = go.Layout(
        paper_bgcolor='rgba(0,0,0,0)', # transparent background
        plot_bgcolor='rgba(0,0,0,0)', # transparent 2nd background
        xaxis =  {'showgrid': False, 'zeroline': False}, # no gridlines
        yaxis = {'showgrid': False, 'zeroline': False}, # no gridlines
    )

    ## Create figure
    fig = go.Figure(layout = layout)
    ## Add all edge traces
    for trace in edge_trace:
        fig.add_trace(trace)
    fig.add_trace(node_trace)
    fig.update_layout(showlegend = False)
    fig.update_xaxes(showticklabels = False)
    fig.update_yaxes(showticklabels = False)
    fig.update_layout(title=title)
    fig.show()